# Distributed VLA Fine-Tuning with Ray Train

## TLDR

You take the streaming LIBERO pipeline from notebook 01 and hand it to **Ray
Train** to fine-tune **PI0.5** — a 3.4-billion-parameter Vision-Language-Action
policy — across all 4 GPUs with PyTorch **DDP**. Ray Train wraps the model for
distributed data-parallel, shards the streaming dataset across workers, reports
synced metrics, and writes a fault-tolerant checkpoint. You write a normal
PyTorch training loop; Ray handles the distribution. Scaling from 4 GPUs to 400
is one number in `ScalingConfig`.

## Introduction

PI0.5 is a flow-matching VLA: a frozen PaliGemma vision-language backbone plus a
small **action expert** that turns the fused representation into robot action
chunks. Fine-tuning the whole 3.4B model on every demonstration is expensive and
unnecessary — here we **freeze the backbone and train only the four action-head
projections** (`action_in_proj`, `action_out_proj`, `time_mlp_in`,
`time_mlp_out`). That keeps the trainable footprint tiny while still adapting the
policy, and it makes the checkpoint small enough to ship to a serving replica in
notebook 03.

The interesting part is not the model — it is that the **training loop is a
plain PyTorch loop**, and Ray Train turns it into a synchronized 4-GPU DDP job
with checkpointing and fault tolerance, with no distributed boilerplate.

## Key concepts used in this notebook

**Ray Train** is Ray's distributed-training library. You pass a
`train_loop_per_worker` function to a `TorchTrainer`; Ray launches one copy per
GPU worker, each running your loop on its own shard of the data.

**`prepare_model()`** wraps your `nn.Module` in PyTorch DDP and moves it to the
worker's GPU — replacing manual `init_process_group` / `DistributedDataParallel`.

**`get_dataset_shard()`** hands each worker its slice of the Ray Data stream —
replacing `DistributedSampler` and manual sharding.

**`train.report(metrics, checkpoint=...)`** reports metrics from every worker
(aggregated by Ray) and persists a checkpoint from rank 0.

**`ScalingConfig(num_workers=N, use_gpu=True)`** is the one knob that sets how
many GPU workers run. 4 here; 400 in production — same loop.

**`FailureConfig(max_failures=1)`** restarts the job from the last checkpoint on
a worker failure — essential for multi-hour runs.

**Gradient accumulation.** An L4 (24 GB) fits a PI0.5 batch of 1. We accumulate
8 micro-batches before an optimizer step, so the effective batch is
`1 × 8 × 4 workers = 32`.

## What you will learn

- Stage a 3.4B model to per-node local disk **once per node** with Ray
- Write a `train_loop_per_worker` and launch it on 4 GPUs with `TorchTrainer`
- Use `prepare_model`, `get_dataset_shard`, and `train.report` instead of
  distributed boilerplate
- Freeze a backbone and train only the action heads
- Produce a fault-tolerant checkpoint on shared storage that notebook 03 serves

## Why Ray Train?

| Challenge | Without Ray Train | With Ray Train |
|---|---|---|
| Multi-GPU DDP | `init_process_group`, device juggling, `DDP(...)` | `prepare_model(policy)` |
| Data sharding | `DistributedSampler`, manual splits | `get_dataset_shard("train")` |
| Checkpoint coordination | rank-0 filesystem dance | `train.report(checkpoint=...)` |
| Fault tolerance | custom retry/restore logic | `FailureConfig(max_failures=1)` |
| Scale 4 → 400 GPUs | rewrite launch + data plumbing | `ScalingConfig(num_workers=400)` |

## Architecture

```
   Ray Data stream (from notebook 01)
            │  get_dataset_shard("train")  → 4 shards
            ▼
  ┌──────────┬──────────┬──────────┬──────────┐
  │ worker 0 │ worker 1 │ worker 2 │ worker 3 │   each: 1 × L4
  │ PI0.5    │ PI0.5    │ PI0.5    │ PI0.5     │   prepare_model → DDP
  │ (frozen  │          │          │          │   train action heads only
  │  backbone│  …       │  …       │  …        │   grad-accum × 8
  │  + heads)│          │          │          │
  └────┬─────┴────┬─────┴────┬─────┴────┬─────┘
       └──────────┴── DDP all-reduce ───┘
                        │ rank 0
                        ▼
        checkpoint_round1/state.pkl  (shared /mnt/cluster_storage)
                        │
                        ▼  served in notebook 03
```

## Cell 1: Configuration

**What you do** — set the dataset/model repos, the shared-storage checkpoint
path, and the training hyper-parameters.

**What to check** — `HF_TOKEN` is set (PI0.5 pulls the gated
`google/paligemma-3b-pt-224` backbone). `MAX_TRAIN_STEPS = 50` keeps this a
smoke run that finishes in minutes — set it to `None` for a full epoch.

**Why it matters** — every scale lever (steps, workers, batch) lives here; the
loop and infrastructure below never change.

In [1]:
import logging, os, shutil, sys, time
from pathlib import Path
import numpy as np
import torch

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s")
log = logging.getLogger("vla_finetune")

HF_DATASET_REPO = "lerobot/libero"
HF_PI05_REPO    = "lerobot/pi05_libero_finetuned"
HF_DATASET_URI  = f"hf://datasets/{HF_DATASET_REPO}"

# pi05_libero_finetuned: 7-D action, 8-D state, cameras image/image2 at 256x256.
LOCAL_MODEL_DIR      = Path("/mnt/local_storage/lerobot/pi05_libero_finetuned")
CLUSTER_STORAGE_ROOT = Path("/mnt/cluster_storage/vla_closed_loop_demo")  # shared FS
CAMERA_RENAME        = {}                       # LIBERO names need no remap

MAX_TRAIN_STEPS = 50      # smoke run; set to None for a full epoch (~12 hrs)

HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "Set HF_TOKEN before running (gated PaliGemma backbone)."

CLUSTER_STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Dataset : {HF_DATASET_URI}")
print(f"Model   : {HF_PI05_REPO} -> {LOCAL_MODEL_DIR}")
print(f"Storage : {CLUSTER_STORAGE_ROOT}")

Dataset : hf://datasets/lerobot/libero
Model   : lerobot/pi05_libero_finetuned -> /mnt/local_storage/lerobot/pi05_libero_finetuned
Storage : /mnt/cluster_storage/vla_closed_loop_demo


## Cell 2: Connect to Ray

**What you do** — connect and thread the environment every GPU worker needs.

**What to check** — 4 GPUs reported. The env vars matter: `TORCHDYNAMO_DISABLE`
(PI0.5 calls `torch.compile`; workers have no C compiler), the `NCCL_*_DISABLE`
flags (force a safe TCP ring on containerized GPUs), and `HF_TOKEN` (gated
backbone on every worker).

**Why it matters** — `working_dir="."` ships `util.py` and `lerobot_datasource.py`
to all workers; the env vars are the difference between a clean DDP run and a
segfault on this cluster image.

In [2]:
import ray

ray.init(
    address="auto",
    runtime_env={
        "working_dir": ".",
        "env_vars": {
            "HF_TOKEN":                  HF_TOKEN,
            "HF_HUB_ENABLE_HF_TRANSFER": "1",
            "PYTORCH_CUDA_ALLOC_CONF":   "expandable_segments:True",
            "TORCHDYNAMO_DISABLE":       "1",   # no C compiler on workers
            "NCCL_P2P_DISABLE":          "1",   # safe TCP ring on containerized GPUs
            "NCCL_SHM_DISABLE":          "1",
            "NCCL_IB_DISABLE":           "1",
        },
    },
    ignore_reinit_error=True,
)
logging.getLogger("ray.data").setLevel(logging.WARNING)
logging.getLogger("openlineage").setLevel(logging.WARNING)

res = ray.cluster_resources()
print(f"Cluster: GPU={int(res['GPU'])}, CPU={int(res['CPU'])}, memory={res['memory']/1e9:.0f} GiB")

2026-06-08 18:55:37,425	INFO worker.py:1821 -- Connecting to existing Ray cluster at address: 10.0.69.43:6379...


2026-06-08 18:55:37,436	INFO worker.py:1998 -- Connected to Ray cluster. View the dashboard at https://session-ih3rjmvr7i1pepvl4xqj5nplaq.i.anyscaleuserdata.com 


2026-06-08 18:55:37,453	INFO packaging.py:392 -- Ignoring upload to cluster for these files: [PosixPath('/home/ray/default/ray_summit_robotics_2026/.gitignore')]


2026-06-08 18:55:37,669	INFO packaging.py:691 -- Creating a file package for local module '.'.


2026-06-08 18:55:37,670	INFO packaging.py:392 -- Ignoring upload to cluster for these files: [PosixPath('/home/ray/default/ray_summit_robotics_2026/.gitignore')]


...

2026-06-08 18:55:37,870	WARNING packaging.py:516 -- File /home/ray/default/ray_summit_robotics_2026/.git/objects/38/tmp_obj_FOjSbQ is very large (35.30MiB). Consider adding this file to the 'excludes' list to skip uploading it: `ray.init(..., runtime_env={'excludes': ['/home/ray/default/ray_summit_robotics_2026/.git/objects/38/tmp_obj_FOjSbQ']})`


2026-06-08 18:55:38,074	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_051d39e5fd739809.zip' (138.64MiB) to Ray cluster...


2026-06-08 18:55:38,725	INFO packaging.py:476 -- Successfully pushed file package 'gcs://_ray_pkg_051d39e5fd739809.zip'.


Cluster: GPU=4, CPU=64, memory=309 GiB


/home/ray/anaconda3/lib/python3.11/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


## Cell 3: Stage the model + open the data stream

**What you do** — stage the PI0.5 checkpoint to **each node's** local disk (it's
loaded from local safetensors by every worker), then open the LIBERO datasource
and extract the normalization `stats` the preprocessor needs.

**What to check** — staging prints `cached`/`downloaded` per node; the dataset
reports 1,693 episodes / 273k frames. **The dataset is never downloaded** — only
the 3.4 GB model is staged; training data streams (notebook 01).

**Why it matters** — this is the scalable split: *artifacts* are staged and
reused per node; *data* is a stream. The same code path works whether LIBERO is
10 GB or 10 TB.

In [3]:
import util
from lerobot_datasource import LeRobotDatasource

util.stage_on_all_nodes(
    ray, lambda: util.stage_model_to_local(HF_PI05_REPO, LOCAL_MODEL_DIR),
    HF_PI05_REPO, LOCAL_MODEL_DIR, log_fn=log.info,
)

source       = LeRobotDatasource(HF_DATASET_URI)
TOTAL_FRAMES = source.meta.total_frames
STATS        = {
    k: {"mean": v["mean"], "std": v["std"]}
    for k, v in source.meta.stats.items()
    if k in ("action", "observation.state")
}
IMAGE_KEYS = [CAMERA_RENAME.get(k, k) for k in source.meta.video_keys]
print(f"Frames streaming: {TOTAL_FRAMES:,}  | cameras: {source.meta.video_keys}")

2026-06-08 18:55:39,984  INFO      Preprocessor id collision: 'io.ray.preprocessors.ordinal_encoder' was already registered by ray.data.preprocessors.encoder.OrdinalEncoder. Overwriting with ray.anyscale.data.preprocessors.turbo_encoder.OrdinalEncoder.


2026-06-08 18:55:39,984  INFO      Preprocessor id collision: 'io.ray.preprocessors.one_hot_encoder' was already registered by ray.data.preprocessors.encoder.OneHotEncoder. Overwriting with ray.anyscale.data.preprocessors.turbo_encoder.OneHotEncoder.


2026-06-08 18:55:39,985  INFO      Preprocessor id collision: 'io.ray.preprocessors.multi_hot_encoder' was already registered by ray.data.preprocessors.encoder.MultiHotEncoder. Overwriting with ray.anyscale.data.preprocessors.turbo_encoder.MultiHotEncoder.


2026-06-08 18:55:39,985  INFO      Preprocessor id collision: 'io.ray.preprocessors.label_encoder' was already registered by ray.data.preprocessors.encoder.LabelEncoder. Overwriting with ray.anyscale.data.preprocessors.turbo_encoder.LabelEncoder.


2026-06-08 18:55:39,986  INFO      Preprocessor id collision: 'io.ray.preprocessors.categorizer' was already registered by ray.data.preprocessors.encoder.Categorizer. Overwriting with ray.anyscale.data.preprocessors.turbo_encoder.Categorizer.


...

2026-06-08 18:55:44,875  INFO        ip-10-0-71-159: cached: /mnt/local_storage/lerobot/pi05_libero_finetuned


2026-06-08 18:55:44,875  INFO        ip-10-0-93-36: cached: /mnt/local_storage/lerobot/pi05_libero_finetuned


2026-06-08 18:55:44,876  INFO        ip-10-0-86-10: cached: /mnt/local_storage/lerobot/pi05_libero_finetuned


2026-06-08 18:55:46,237  INFO      LeRobotDatasource ready: 1 roots, 273465 total frames, 2 cameras ['observation.images.image', 'observation.images.image2'], mode='file_group'


Frames streaming: 273,465  | cameras: ['observation.images.image', 'observation.images.image2']


## Cell 4: The preprocessing pipeline (from notebook 01)

**What you do** — rebuild the lazy `read → rename → transpose` pipeline. This is
the same `build_libero_dataset` you stepped through in notebook 01.

**What to check** — it returns instantly (lazy); nothing executes until the
trainer pulls shards.

**Why it matters** — the data layer is decoupled from training. The trainer just
asks for batches; Ray Data streams and preprocesses them on the CPU pool.

In [4]:
def rename_columns(row, rename):
    return {rename.get(k, k): v for k, v in row.items()}


def transpose_images(batch, camera_keys):
    """HWC uint8 -> CHW float32."""
    out = dict(batch)
    for key in camera_keys:
        out[key] = np.transpose(np.stack(list(batch[key])), (0, 3, 1, 2)).astype(np.float32)
    return out


def build_libero_dataset():
    return (
        ray.data.read_datasource(source)
        .map(rename_columns, fn_args=(CAMERA_RENAME,))
        .map_batches(transpose_images, batch_size=32, fn_args=(IMAGE_KEYS,))
    )

print(build_libero_dataset())

2026-06-08 18:55:46,254  INFO      37 tasks, 273465 total frames, 1 roots, 2 cameras


2026-06-08 18:55:46,329  INFO      1 tasks, 273465 total frames, 1 roots, 2 cameras


MapBatches(transpose_images)
+- Map(rename_columns)
   +- Dataset(
         num_rows=273465,
         schema={
            observation.state: list<element: float>,
            action: list<element: float>,
            timestamp: float,
            frame_index: int64,
            episode_index: int64,
            index: int64,
            task_index: int64,
            observation.i...: ArrowVariableShapedTensorType(ndim=3, dtype=uint8),
            observation.i...: ArrowVariableShapedTensorType(ndim=3, dtype=uint8),
            task: string,
            dataset_index: int32
         }
      )


## Cell 5: The per-worker training loop

**What you do** — define `train_loop_per_worker`, the function Ray Train runs on
every GPU. It loads PI0.5 (freezing all but the action heads via
`util.load_pi05_policy`), wraps it with `prepare_model` (DDP), builds the
preprocessor, then iterates its **dataset shard** with gradient accumulation,
reporting metrics and a rank-0 checkpoint each epoch.

**What to check** — the three Ray Train touchpoints marked in comments:
`prepare_model` (DDP), `get_dataset_shard` (sharding), `train.report`
(metrics + checkpoint). Everything else is ordinary PyTorch.

**Why it matters** — this is the whole point: a normal loop becomes a
distributed, fault-tolerant, checkpointing 4-GPU job with three Ray calls.

In [5]:
import ray.train
import ray.train.torch


def train_loop_per_worker(config):
    from lerobot.policies.factory import make_pre_post_processors

    device = torch.device("cuda")
    policy = util.load_pi05_policy(LOCAL_MODEL_DIR)
    policy = ray.train.torch.prepare_model(policy)              # <-- RAY TRAIN: DDP wrap

    optimizer = torch.optim.AdamW(
        [p for p in policy.parameters() if p.requires_grad],
        lr=config.get("lr", 5e-5),
    )
    scaler = torch.amp.GradScaler("cuda")

    checkpoint = ray.train.get_checkpoint()                     # <-- RAY TRAIN: fault tolerance
    start_epoch, step = (util.load_checkpoint(checkpoint, policy, optimizer, scaler)
                         if checkpoint else (0, 0))

    preprocessor, _ = make_pre_post_processors(
        policy.module.config,
        pretrained_path=str(LOCAL_MODEL_DIR),
        dataset_stats=config["stats"],
    )

    batch_size      = int(config.get("batch_size", 1))
    grad_accum      = int(config.get("grad_accum", 8))
    num_epochs      = int(config.get("num_epochs", 1))
    max_len         = int(config.get("max_len", 512))
    max_train_steps = config.get("max_train_steps")
    num_workers     = ray.train.get_context().get_world_size()
    rank            = ray.train.get_context().get_world_rank()
    scheduler       = util.build_lr_scheduler(optimizer, config, num_workers, last_step=step)
    shard           = ray.train.get_dataset_shard("train")      # <-- RAY DATA: per-worker shard

    for epoch in range(start_epoch, num_epochs):
        optimizer.zero_grad(set_to_none=True)
        accum = 0
        loss_sum, loss_count = 0.0, 0

        for batch in shard.iter_torch_batches(
            batch_size=batch_size,
            collate_fn=util.NumpyToTorchCollate(device),
        ):
            loss_val = util.train_step(policy, batch, preprocessor, max_len, grad_accum, scaler)
            step += 1; accum += 1
            loss_sum += loss_val; loss_count += 1

            if accum % grad_accum == 0:
                util.optimizer_step(policy, optimizer, scaler, scheduler)
                accum = 0

            if step % 10 == 0 and rank == 0:
                log.info("epoch=%d  step=%d  loss=%.4f  lr=%.2e",
                         epoch, step, loss_val, scheduler.get_last_lr()[0])

            if max_train_steps and step >= max_train_steps:
                break

        if accum > 0:
            util.optimizer_step(policy, optimizer, scaler, scheduler)

        avg_loss = loss_sum / max(loss_count, 1)
        metrics  = {"epoch": epoch, "steps": step,
                    "loss": avg_loss, "lr": scheduler.get_last_lr()[0]}

        if rank == 0:
            ckpt = util.make_checkpoint(
                policy, optimizer, scaler, epoch, step, config["stats"],
                base_model_repo=HF_PI05_REPO, camera_rename=CAMERA_RENAME,
            )
            ray.train.report(metrics, checkpoint=ckpt)          # <-- RAY TRAIN: synced report
        else:
            ray.train.report(metrics)

        if max_train_steps and step >= max_train_steps:
            break

## Cell 6: Launch helper

**What you do** — wrap `TorchTrainer` in `run_training`: it sets the
`ScalingConfig` (4 GPU workers), `RunConfig` (storage + `FailureConfig` +
checkpoint retention), passes the dataset, calls `.fit()`, then copies the
resulting checkpoint to a **stable path** (`checkpoint_round1/state.pkl`) so the
serving replica in notebook 03 can read it without knowing Ray Train's internal
layout.

**What to check** — `num_workers=4, use_gpu=True`; the effective batch is
`1 × grad_accum(8) × 4 = 32`.

**Why it matters** — scaling to more GPUs is editing `num_workers`; nothing else
in this function or the loop changes.

In [6]:
def run_training(ds, round_name):
    """Run TorchTrainer on `ds`, copy checkpoint to a stable path, return path + metrics."""
    CLUSTER_STORAGE_ROOT.mkdir(parents=True, exist_ok=True)

    result = ray.train.torch.TorchTrainer(
        train_loop_per_worker=train_loop_per_worker,
        train_loop_config={
            "stats":           STATS,
            "total_rows":      TOTAL_FRAMES,
            "num_epochs":      1,
            "batch_size":      1,      # L4 (24 GB) — bs=1 is the largest that fits
            "grad_accum":      8,      # effective batch = 1 * 8 * 4 workers = 32
            "lr":              5e-5,
            "warmup_frac":     0.1,
            "max_len":         512,
            "max_train_steps": MAX_TRAIN_STEPS,
        },
        scaling_config=ray.train.ScalingConfig(num_workers=4, use_gpu=True),
        run_config=ray.train.RunConfig(
            name=f"vla-finetune-{round_name}",
            storage_path=str(CLUSTER_STORAGE_ROOT),
            failure_config=ray.train.FailureConfig(max_failures=1),
            checkpoint_config=ray.train.CheckpointConfig(num_to_keep=1),
        ),
        datasets={"train": ds},
    ).fit()

    checkpoint_path = CLUSTER_STORAGE_ROOT / f"checkpoint_{round_name}" / "state.pkl"
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    with result.checkpoint.as_directory() as d:
        shutil.copy2(os.path.join(d, "state.pkl"), checkpoint_path)

    log.info("[%s] checkpoint -> %s", round_name, checkpoint_path)
    log.info("[%s] metrics: %s", round_name, result.metrics)
    return checkpoint_path, result.metrics

## Cell 7: Fine-tune PI0.5 on 4 GPUs

**What you do** — launch the distributed run. Ray Train spins up 4 GPU workers,
each loads PI0.5, wraps it in DDP, and trains on its shard of the LIBERO stream.

**What to check** — `(RayTrainWorker ...)` log lines show `epoch=0 step=N
loss=...` from rank 0 every 10 steps, then a checkpoint is written. With
`MAX_TRAIN_STEPS=50` this is a couple of minutes after model load.

**Why it matters** — this is the full distributed fine-tune. Every later
notebook reuses this exact `TorchTrainer` pattern.

> **Re-running note:** Ray Train resumes from `storage_path` when a run of the
> same `name` already has a checkpoint there. On a fresh cluster this trains
> from scratch (as shown below); to force a fresh run on a populated path,
> clear that run directory first.

In [7]:
ckpt_path, metrics = run_training(build_libero_dataset(), "round1")
print("\nCheckpoint :", ckpt_path)
print("Final metrics:", metrics)

2026-06-08 18:55:46,557  INFO      37 tasks, 273465 total frames, 1 roots, 2 cameras


(TrainController pid=92892) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=92892) Attempting to start training worker group of size 4 with the following resources: [{'GPU': 1}] * 4


(RayTrainWorker pid=32715, ip=10.0.71.159) Setting up process group for: env:// [rank=0, world_size=4]
(TrainController pid=92892) Started training worker group of size 4: 
(TrainController pid=92892) - (ip=10.0.71.159, pid=32715) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=92892) - (ip=10.0.86.10, pid=33629) world_rank=1, local_rank=0, node_rank=1
(TrainController pid=92892) - (ip=10.0.93.36, pid=30691) world_rank=2, local_rank=0, node_rank=2
(TrainController pid=92892) - (ip=10.0.65.164, pid=33703) world_rank=3, local_rank=0, node_rank=3
(TrainController pid=92892) [State Transition] SCHEDULING -> RUNNING.


(RayTrainWorker pid=32715, ip=10.0.71.159) The PI05 model is a direct port of the OpenPI implementation. 
(RayTrainWorker pid=32715, ip=10.0.71.159) This implementation follows the original OpenPI structure for compatibility. 
(RayTrainWorker pid=32715, ip=10.0.71.159) Original implementation: https://github.com/Physical-Intelligence/openpi


(RayTrainWorker pid=30691, ip=10.0.93.36) Enabled gradient checkpointing for PI05Pytorch model
(RayTrainWorker pid=33629, ip=10.0.86.10) The PI05 model is a direct port of the OpenPI implementation.  [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(RayTrainWorker pid=33629, ip=10.0.86.10) This implementation follows the original OpenPI structure for compatibility.  [repeated 3x across cluster]
(RayTrainWorker pid=33629, ip=10.0.86.10) Original implementation: https://github.com/Physical-Intelligence/openpi [repeated 3x across cluster]


...

(SplitCoordinator pid=93605) ✔️  Dataset train_78_0 execution finished in 44.11 seconds
(SplitCoordinator pid=93605) INFO:openlineage.client.client:OpenLineageClient will use `composite` transport
(SplitCoordinator pid=93605) INFO:openlineage.client.transport.composite:Stopping OpenLineage CompositeTransport emission after the first successful delivery because `continue_on_success=False`. Transport that emitted the event: <HttpTransport(name=first, kind=http, priority=1)>


(TrainController pid=92892) [State Transition] SHUTTING_DOWN -> FINISHED.


2026-06-08 18:57:43,888  INFO      [round1] checkpoint -> /mnt/cluster_storage/vla_closed_loop_demo/checkpoint_round1/state.pkl


2026-06-08 18:57:43,888  INFO      [round1] metrics: {'epoch': 0, 'steps': 50, 'loss': 2.588297712802887, 'lr': 4.098360655737705e-07}



Checkpoint : /mnt/cluster_storage/vla_closed_loop_demo/checkpoint_round1/state.pkl
Final metrics: {'epoch': 0, 'steps': 50, 'loss': 2.588297712802887, 'lr': 4.098360655737705e-07}


## Cell 8: Confirm the checkpoint

**What you do** — verify the checkpoint landed on shared storage and peek at its
contents.

**What to check** — `state.pkl` exists and contains the trained head weights
plus the dataset `stats` (so the serving preprocessor can be rebuilt) and
breadcrumbs (`base_model_repo`, `step`, `epoch`). Note we load it with a
CPU-mapping unpickler: training ran on GPU workers so the tensors are tagged
CUDA, but this notebook's kernel runs on the CPU-only head node.

**Why it matters** — this small, self-describing checkpoint is the hand-off to
notebook 03: the Ray Serve replica loads it onto a GPU and serves predictions
over HTTP.

In [8]:
import io, pickle, torch


class _CPUUnpickler(pickle.Unpickler):
    """Load a CUDA-saved checkpoint on the CPU-only head node.

    The trainer ran on GPU workers, so the checkpoint's tensors are tagged
    CUDA. This notebook's kernel is on the CPU head, so we remap storages to
    CPU on load — the same trick policy_server.py uses to inspect checkpoints.
    """
    def find_class(self, module, name):
        if module == "torch.storage" and name == "_load_from_bytes":
            return lambda b: torch.load(io.BytesIO(b), map_location="cpu", weights_only=False)
        return super().find_class(module, name)


print("exists:", ckpt_path.exists(), "| size:", f"{ckpt_path.stat().st_size/1e6:.1f} MB")
with open(ckpt_path, "rb") as f:
    state = _CPUUnpickler(f).load()
print("keys           :", list(state.keys()))
print("trained tensors:", len(state["model"]))
print("stats keys     :", list(state["stats"].keys()))
print("step / epoch   :", state["step"], "/", state["epoch"])

exists: True | size: 26.0 MB
keys           : ['model', 'optim', 'scaler', 'epoch', 'step', 'stats', 'base_model_repo', 'camera_rename']
trained tensors: 8
stats keys     : ['action', 'observation.state']
step / epoch   : 50 / 0


## Conclusion

You fine-tuned a 3.4B-parameter PI0.5 policy across 4 GPUs with **Ray Train** —
DDP via `prepare_model`, streaming shards via `get_dataset_shard`, and a
fault-tolerant checkpoint via `train.report` — wrapped around an ordinary
PyTorch loop. Only the action heads were trained; the checkpoint is small and
self-describing.

**Ray primitives used:** `TorchTrainer`, `prepare_model`, `get_dataset_shard`,
`train.report`, `ScalingConfig`, `FailureConfig`, `CheckpointConfig`.

**Scaling levers:** `ScalingConfig(num_workers=N)` (4 → 400 GPUs);
`MAX_TRAIN_STEPS=None` (full epoch); larger `batch_size` on bigger GPUs.

Next, **`03_serving_and_sim_eval.ipynb`** loads this checkpoint into **Ray
Serve**, then fans out **Isaac Lab** simulation rollouts as Ray tasks that query
the policy over HTTP — and closes the loop by folding the sim data back into
training.